In [1]:
import os

import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Read the file as a text lines.

In [2]:
df = spark.read.text(os.path.expanduser("~/data/OneMillionRows.csv"))

## Skip header line

In [3]:
df = df.filter(~F.col("value").contains("boundingbox,requeststarttime"))

## Parse the line and pull the columns.

In [4]:
df = df.select(
    F.regexp_extract(F.col("value"), r"\[(.*?)\]", 1).alias("bbox_coords"),
    F.regexp_extract(F.col("value"), r"\],(.*)$", 1).alias("request_time"),
)

## Deconstruct the columns.

In [5]:
bbox_split = F.split(F.col("bbox_coords"), ",")
df = df.select(
    F.trim(bbox_split.getItem(0)).cast(T.DoubleType()).alias("xmin"),
    F.trim(bbox_split.getItem(1)).cast(T.DoubleType()).alias("ymin"),
    F.trim(bbox_split.getItem(2)).cast(T.DoubleType()).alias("xmax"),
    F.trim(bbox_split.getItem(3)).cast(T.DoubleType()).alias("ymax"),
    F.trim(F.col("request_time")).cast(T.TimestampType()).alias("request_time"),
)

## Show sample values.

In [6]:
df.limit(10).show(truncate=False)

+--------+-------+--------+------+-----------------------+
|xmin    |ymin   |xmax    |ymax  |request_time           |
+--------+-------+--------+------+-----------------------+
|-76.994 |39.808 |-76.983 |39.811|2024-04-06 23:59:59.685|
|-119.796|36.739 |-119.785|36.822|2024-04-07 00:00:00.591|
|-73.923 |40.696 |-73.507 |40.737|2024-04-07 00:00:00.971|
|-82.591 |28.469 |-82.59  |28.471|2024-04-07 00:00:01.214|
|-74.165 |40.72  |-74.142 |40.733|2024-04-07 00:00:00.683|
|-77.054 |40.075 |-77.028 |40.108|2024-04-07 00:00:01.26 |
|-58.946 |-34.774|-58.475 |-34.16|2024-04-07 00:00:00.291|
|-58.946 |-34.81 |-58.492 |-34.16|2024-04-07 00:00:01.134|
|-74.173 |40.729 |-74.166 |40.756|2024-04-07 00:00:01.591|
|-88.942 |40.487 |-88.942 |40.487|2024-04-07 00:00:03.123|
+--------+-------+--------+------+-----------------------+



## Define the EMEA region.

In [7]:
xmin, ymin, xmax, ymax = (
    -45.363622663867915,
    9.419412209494896,
    89.89250713547236,
    73.70242597609474,
)

# Cont. USA
xmin, ymin, xmax, ymax = (
    -135.26809221776833,
    3.81161827929283,
    -51.44552106252448,
    55.47353704751483,
)

In [8]:
df2 = (
    df
    # Find valid non-empty extents.
    .filter((F.col("xmax") > F.col("xmin")) & (F.col("ymax") > F.col("ymin")))
    # Find rows fully inside the region.
    .filter(
        (F.col("xmin") > F.lit(xmin))
        & (F.col("xmax") < F.lit(xmax))
        & (F.col("ymin") > F.lit(ymin))
        & (F.col("ymax") < F.lit(ymax))
    ).withColumn(
        "geom",
        S.st_simplify(
            S.st_rect(
                "xmin",
                "ymin",
                "xmax",
                "ymax",
            ),
            4326,
        ),
    )
    # .withColumn("miid", F.monotonically_increasing_id())
    # .localCheckpoint()
)

In [9]:
# df2.count()
# df2.select(
#     F.min("xmin"),
#     F.min("ymin"),
#     F.max("xmax"),
#     F.max("ymax"),
# ).show(truncate=False)

In [10]:
# df.write.parquet(os.path.expanduser("~/OneMillion.prq"), mode="overwrite")

## Convert each geometry to a list of cells and get the count by cell.

In [11]:
cell = 1.0

qr_count = (
    df2.select(S.qr_list_explode("geom", cell)).groupBy("qr").count().localCheckpoint()
)

## Get the mean and sd to calculate the z-score of each cell based on the count.

In [12]:
rows = (
    qr_count.select("count")
    .summary("mean", "stddev")
    .select(F.col("count").alias("n"))
    .collect()
)
mu = F.lit(float(rows[0].n))
sd = F.lit(float(rows[1].n))
mu, sd

(Column<'2208.139127855236'>, Column<'4563.8978876945'>)

## Calculate the z-score and convert it to a pandas DF.

In [13]:
zmin = -3.5
zmax = 3.5

pdf = (
    qr_count.select(
        S.qr_asgeom("qr", cell).alias("geometry"),
        F.col("count").cast(T.IntegerType()).alias("count"),
        ((F.col("count") - mu) / sd).alias("z"),
    )
    .orderBy("z")
    .withColumn("z", F.greatest(F.lit(zmin), F.least(F.lit(zmax), "z")))
    .toPandas()
)

## Convert Pandas DF to GeoPandas DF.

In [14]:
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:4326")
# Enable spatial index.
gdf.sindex;

## Optional - Save to GeoPandas DF to a FileGDB to be visualized in Pro.

In [15]:
# layer_name = "qr_count"
# geodatabase_path = os.path.expanduser(f"~/{layer_name}.gdb")
# gdf.to_file(geodatabase_path, layer=layer_name, driver="OpenFileGDB")

## Let's view the result in the notebook.

In [16]:
gdf.explore(
    "z",
    cmap="coolwarm",
    vmin=zmin,
    vmax=zmax,
    tiles=xyz.Esri.WorldGrayCanvas,
)